# Regression: Submission Skeleton (ส่งโมเดลเดียว, from scratch)

**วิธีใช้ตอนสอบ**
1. ทำการเทียบโมเดล/จูนใน `regression_exam.ipynb` จนรู้ว่าตัวไหนชนะ และได้ hyperparameter
2. เปิดไฟล์นี้ แก้ `TARGET` ใน cell Config
3. เปิด `regression_model_snippets.ipynb` ก้อป cell ของโมเดลที่เลือก ไปวาง **แทน cell "MODEL SLOT"** (cell เดียว)
4. แก้ hyperparameter ใน `make_model()` ให้เป็นค่าที่จูนได้
5. Run all → ได้ผล R²/RMSE/MAE + ฟังก์ชัน `predict_new(df_new)` สำหรับข้อมูลใหม่

**สัญญาระหว่าง skeleton กับ MODEL SLOT:** slot ต้องกำหนด 2 อย่าง คือ `FEATURES` (`'scaled'` หรือ `'raw'`) และ `make_model()` ที่คืนโมเดลที่มี `.fit(X, y)` / `.predict(X)`

## 0. โหลดข้อมูล

In [ ]:
import pandas as pd

# แบบ 1: ไฟล์ csv อยู่ใน Colab แล้ว
df = pd.read_csv('data.csv')

# แบบ 2: ต้องอัปโหลดเอง
# from google.colab import files
# files.upload()
# df = pd.read_csv('ชื่อไฟล์.csv')

# แบบ 3: อาจารย์ให้ dataframe มาแล้ว (เช่นชื่อ data) -> df = data.copy()

df.head()

## 1. Config

In [ ]:
TARGET        = 'ชื่อคอลัมน์เป้าหมาย'   # <-- แก้ตรงนี้
DROP_COLS     = []           # คอลัมน์ที่จะทิ้ง เช่น ['id', 'name']
TEST_SIZE     = 0.2
SEED          = 42
MAX_NULL_COL  = 0.50         # คอลัมน์ null เกินนี้ -> ทิ้ง
ADD_INDICATOR = 0.05         # คอลัมน์ null เกินนี้ -> เพิ่ม flag _missing
USE_LOG_TARGET = False       # True = เทรนบน log1p(y) (ผลวัดยังเป็นหน่วยเดิม)

## 2. Data prep (ไม่ต้องแก้)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

data = df.drop_duplicates().copy()
data = data.dropna(subset=[TARGET])
data = data.drop(columns=[c for c in DROP_COLS if c in data.columns])

feat = [c for c in data.columns if c != TARGET]
ratio = data[feat].isna().mean()
feat = [c for c in feat if ratio[c] <= MAX_NULL_COL]                    # ทิ้งคอลัมน์ null เยอะ
ind_cols = [c for c in feat if data[c].isna().mean() > ADD_INDICATOR]   # คอลัมน์ที่จะเพิ่ม flag
raw_feat = feat.copy()                                                  # คอลัมน์ที่ข้อมูลใหม่ต้องมี
cat_cols = data[feat].select_dtypes(exclude='number').columns.tolist()

def clean_X(d_):
    X_ = d_[raw_feat].copy()
    for c in ind_cols:
        X_[c + '_missing'] = X_[c].isna().astype(int)
    for c in cat_cols:
        X_[c] = X_[c].astype(object).fillna('missing').astype(str)
    return X_

X_all = pd.get_dummies(clean_X(data), drop_first=True, dtype=float)
dummy_cols = X_all.columns
X_train, X_test, y_train, y_test = train_test_split(X_all, data[TARGET], test_size=TEST_SIZE, random_state=SEED)

med = X_train.median()                                     # impute ด้วย median ของ train
X_train, X_test = X_train.fillna(med), X_test.fillna(med)

Xtr, Xte = X_train.to_numpy(float), X_test.to_numpy(float)
ytr, yte = y_train.to_numpy(float), y_test.to_numpy(float)
mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
sd[sd == 0] = 1
Xtr_s, Xte_s = (Xtr - mu) / sd, (Xte - mu) / sd            # standardize ด้วยสถิติของ train

def to_matrix(d_):
    """แปลง dataframe ดิบ (ข้อมูลใหม่) -> (raw, scaled) ด้วยกระบวนการเดียวกับ train"""
    X_ = pd.get_dummies(clean_X(d_), drop_first=True, dtype=float).reindex(columns=dummy_cols, fill_value=0.0).fillna(med)
    raw = X_.to_numpy(float)
    return raw, (raw - mu) / sd

print('train:', Xtr.shape, ' test:', Xte.shape)

## 3. ตัวช่วย (ไม่ต้องแก้): metrics + log-target wrapper

In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def evaluate(name, y_true, y_pred):
    r2, rmse, mae = r2_score(y_true, y_pred), np.sqrt(mean_squared_error(y_true, y_pred)), mean_absolute_error(y_true, y_pred)
    print(f'{name:<10} R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}')
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae}

class LogTarget:
    """เทรนบน log1p(y) แต่ predict คืนหน่วยเดิม"""
    def __init__(self, model): self.model = model
    def fit(self, X, y):
        self.model.fit(X, np.log1p(y)); return self
    def predict(self, X):
        return np.expm1(np.clip(self.model.predict(X), -20, 50))
    def __getattr__(self, name):
        return getattr(self.model, name)

## 4. MODEL SLOT  ← วาง snippet โมเดลที่เลือกแทน cell นี้ทั้ง cell
(ค่าเริ่มต้นเป็น Ridge เพื่อให้ skeleton รันได้ทันที)

In [ ]:
# ===== MODEL: Ridge Regression =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class LinearRegressionScratch:
    def __init__(self, lam=0.0):
        self.lam = lam

    def fit(self, X, y):
        A = np.c_[np.ones(len(X)), X]               # เติมคอลัมน์ 1 = intercept
        I = np.eye(A.shape[1]); I[0, 0] = 0         # ไม่ penalize intercept
        self.w = np.linalg.pinv(A.T @ A + self.lam * I) @ A.T @ y
        return self

    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w

FEATURES = 'scaled'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return LinearRegressionScratch(lam=10.0)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (lam: ค่ามาก = regularize แรง (ลอง 0.1, 1, 10, 100))

## 5. Train + Evaluate

In [ ]:
A_tr, A_te = (Xtr_s, Xte_s) if FEATURES == 'scaled' else (Xtr, Xte)

if USE_LOG_TARGET and ytr.min() <= -1:
    print('y มีค่า <= -1 ใช้ log1p ไม่ได้ -> ปิด USE_LOG_TARGET'); USE_LOG_TARGET = False

model = make_model()
if USE_LOG_TARGET:
    model = LogTarget(model)
model.fit(A_tr, ytr)

print('features:', FEATURES, '| log target:', USE_LOG_TARGET)
evaluate('train', ytr, model.predict(A_tr))
res = evaluate('TEST', yte, model.predict(A_te))      # <- ตัวเลขที่ใช้รายงาน

In [ ]:
import matplotlib.pyplot as plt

pred = model.predict(A_te)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(yte, pred, alpha=0.6); lo, hi = min(yte.min(), pred.min()), max(yte.max(), pred.max())
ax[0].plot([lo, hi], [lo, hi], 'r--'); ax[0].set_xlabel('Actual'); ax[0].set_ylabel('Predicted'); ax[0].set_title('Actual vs Predicted')
ax[1].scatter(pred, yte - pred, alpha=0.6); ax[1].axhline(0, color='r', linestyle='--')
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('Residual'); ax[1].set_title('Residual plot')
plt.show()

## 6. ทำนายข้อมูลใหม่ (ถ้าอาจารย์ให้ test set แยกมา)

In [ ]:
def predict_new(df_new):
    """รับ dataframe ดิบ (ไม่ต้องมีคอลัมน์ target) -> คืน array ของค่าทำนาย"""
    raw, sc = to_matrix(df_new)
    return model.predict(sc if FEATURES == 'scaled' else raw)

# เช็คความถูกต้อง: ส่งแถวดิบของ test เข้าไป ต้องได้ผลเท่ากับข้างบน
chk = predict_new(data.loc[X_test.index])
print('predict_new ตรงกับผล test:', np.allclose(chk, model.predict(A_te)))

# ถ้ามีไฟล์ test ของอาจารย์:
# new_df = pd.read_csv('test.csv')
# out = pd.DataFrame({'prediction': predict_new(new_df)})
# out.to_csv('predictions.csv', index=False)